# Retail Demand Forecasting — Business Insights & Recommendations

## Objective

The aim of this notebook is to translate the forecasting results into practical planning insights.

I will:

- review forecast accuracy across the selected products
- identify which products are easier or harder to forecast
- highlight demand patterns that could affect stock planning
- consider the impact of demand volatility and seasonal trading periods
- produce clear recommendations for inventory and demand planning

In [14]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_PROCESSED = Path("../data/processed")

MODEL_FILE = DATA_PROCESSED / "model_summary.csv"
FORECAST_FILE = DATA_PROCESSED / "final_forecast_results.csv"
WEEKLY_FILE = DATA_PROCESSED / "weekly_forecast_data.csv"

model_summary = pd.read_csv(MODEL_FILE)

forecast_results = pd.read_csv(
    FORECAST_FILE,
    parse_dates=["invoice_date"]
)

forecast_data = pd.read_csv(
    WEEKLY_FILE,
    parse_dates=["invoice_date"]
)

In [11]:
print("Model summary rows:", len(model_summary))
print("Forecast result rows:", len(forecast_results))

print("\nModel summary:")
display(model_summary)

Model summary rows: 5
Forecast result rows: 60

Model summary:


,stock_code,best_model,baseline_WMAPE,moving_avg_WMAPE,seasonal_WMAPE,exp_smoothing_WMAPE,holt_WMAPE,best_WMAPE
0,21212,Holt trend,80.339917,43.959577,152.947481,49.940674,41.553094,41.55
1,84077,Exponential smoothing,52.272874,44.400026,117.351215,38.395442,44.080396,38.40
2,84879,Exponential smoothing,28.238462,35.537190,82.598199,26.622713,29.063127,26.62
3,85099B,Holt trend,46.465849,43.019264,44.854641,42.702093,41.750200,41.75
4,85123A,Naive baseline,42.049470,44.216726,88.127208,42.665976,46.892709,42.05


## Forecast Accuracy and Planning Risk

Forecast accuracy varies across the five selected products.

To make the results easier to interpret from a planning perspective, I calculated the best WMAPE achieved for each product and grouped products into simple forecast-risk categories.

Lower WMAPE means the forecast is generally more accurate, while higher WMAPE indicates greater uncertainty and a higher risk of over- or under-stocking.

In [19]:
wmape_columns = [
    "baseline_WMAPE",
    "moving_avg_WMAPE",
    "seasonal_WMAPE",
    "exp_smoothing_WMAPE",
    "holt_WMAPE"
]

model_summary["best_WMAPE"] = (
    model_summary[wmape_columns]
    .min(axis=1)
    .round(2)
)

model_summary[
    ["stock_code", "best_model", "best_WMAPE"]
]

def forecast_risk(wmape):
    if wmape < 30:
        return "Lower"
    elif wmape < 40:
        return "Moderate"
    else:
        return "Higher"

model_summary["forecast_risk"] = (
    model_summary["best_WMAPE"]
    .apply(forecast_risk)
)

model_summary[
    [
        "stock_code",
        "best_model",
        "best_WMAPE",
        "forecast_risk"
    ]
]

,stock_code,best_model,best_WMAPE,forecast_risk
0,21212,Holt trend,41.55,Higher
1,84077,Exponential smoothing,38.40,Moderate
2,84879,Exponential smoothing,26.62,Lower
3,85099B,Holt trend,41.75,Higher
4,85123A,Naive baseline,42.05,Higher


### Forecast Accuracy Observation

Forecast accuracy differs materially across the five selected products.

`84879` has the lowest forecast error and can be planned with greater confidence, while `21212`, `85099B` and `85123A` remain more difficult to predict.

Higher forecast error increases the risk of both over-stocking and stockouts, so these products should receive more cautious inventory planning and closer forecast monitoring.

### Recommendation 1 — Use Forecast Risk to Guide Inventory Decisions

**Priority:** High

**Recommendation:**  
Use forecast reliability to guide inventory decisions. Products with higher forecast error should be reviewed more frequently and planned with greater caution to reduce the risk of stockouts or excess inventory.

**Owner:**  
Demand Planning / Inventory Planning

**Expected Impact:**  
Reduce stockout and excess inventory risk by aligning stock decisions with the reliability of each product forecast.

**Metric to Track:**  
- Forecast error
- Stockout rate
- Excess stock / aged inventory
- Forecast bias

## Demand Volatility and Peak Risk

Several selected products show large short-term demand spikes even when their normal weekly demand is relatively stable.

These peaks are difficult for simple time-series models to predict and can create stockout risk if inventory planning is based only on the average forecast.

In [15]:
volatility_summary = (
    forecast_data
    .groupby("stock_code")
    .agg(
        average_weekly_demand=("weekly_demand", "mean"),
        max_weekly_demand=("weekly_demand", "max"),
        std_weekly_demand=("weekly_demand", "std")
    )
    .reset_index()
)

volatility_summary["peak_to_average_ratio"] = (
    volatility_summary["max_weekly_demand"]
    / volatility_summary["average_weekly_demand"]
)

volatility_summary[
    [
        "stock_code",
        "average_weekly_demand",
        "max_weekly_demand",
        "peak_to_average_ratio"
    ]
].sort_values(
    "peak_to_average_ratio",
    ascending=False
)

,stock_code,average_weekly_demand,max_weekly_demand,peak_to_average_ratio
2,84879,753.481132,5584,7.410935
1,84077,1000.405660,5328,5.325840
4,85123A,884.613208,3328,3.762096
0,21212,893.773585,2731,3.055584
3,85099B,905.575472,2569,2.836870


### Demand Volatility Observation

The selected products differ substantially in peak-demand risk.

`84879` has the highest peak-to-average ratio, with its largest week reaching more than seven times its average weekly demand. `84077` also shows significant peak behaviour.

This means forecast accuracy should not be considered in isolation. A product can have relatively good average forecast performance while still being exposed to occasional demand surges that create stockout risk.

### Recommendation 2 — Protect High-Volatility Products

**Priority:** High

**Recommendation:**  
Hold additional stock protection for products that experience large demand surges, particularly `84879` and `84077`, rather than relying only on their average weekly forecast.

**Owner:**  
Demand Planning / Inventory Planning

**Expected Impact:**  
Improve availability during unexpected demand peaks and reduce the risk of lost sales from stockouts.

**Metric to Track:**  
- Peak demand risk
- Stockout rate
- Service level / fill rate
- Forecast error during peak weeks

## Seasonal Planning Risk

The exploratory analysis showed a recurring increase in demand during the later months of the year.

Demand generally strengthens from late summer into autumn, with September, October and November showing consistently higher volumes. This creates additional planning pressure ahead of the Christmas trading period.

In [17]:
seasonal_summary["month"] = pd.to_datetime(
    seasonal_summary["month"],
    format="%m"
).dt.month_name()

seasonal_summary["average_weekly_demand"] = (
    seasonal_summary["average_weekly_demand"]
    .round(0)
    .astype(int)
)

seasonal_summary

,month,average_weekly_demand
0,January,604
1,February,663
2,March,851
3,April,926
4,May,890
5,June,663
6,July,803
7,August,913
8,September,878
9,October,1064


### Seasonal Demand Observation

The five selected products also show stronger demand toward the end of the year.

Average weekly demand increases substantially in October and reaches its highest level in November. This supports the wider seasonal pattern identified during the exploratory analysis and reinforces the need to prepare inventory ahead of the autumn and pre-Christmas trading period.

### Recommendation 3 — Prepare Earlier for Peak Season

**Priority:** Medium

**Recommendation:**  
Increase forecast reviews and inventory planning ahead of the September-to-November demand increase so stock is positioned before the seasonal peak begins.

**Owner:**  
Demand Planning / Inventory Planning / Purchasing

**Expected Impact:**  
Improve product availability during peak trading periods and reduce reactive replenishment.

**Metric to Track:**  
- Peak-season forecast error
- Service level / fill rate
- Stockout rate
- Inventory cover

In [20]:
recommendations = pd.DataFrame({
    "priority": ["High", "High", "Medium"],
    "recommendation": [
        "Use forecast-risk categories to guide inventory decisions",
        "Apply additional stock protection to high-volatility products",
        "Prepare inventory earlier for the autumn and pre-Christmas demand increase"
    ],
    "owner": [
        "Demand Planning / Inventory Planning",
        "Demand Planning / Inventory Planning",
        "Demand Planning / Inventory Planning / Purchasing"
    ],
    "expected_impact": [
        "Reduce stockout and excess inventory risk",
        "Improve availability during sudden demand surges",
        "Improve product availability during peak trading periods"
    ],
    "metric_to_track": [
        "WMAPE, stockout rate, excess stock, forecast bias",
        "Peak-to-average ratio, stockout rate, fill rate",
        "Peak-season WMAPE, fill rate, stockout rate, inventory cover"
    ]
})

recommendations

,priority,recommendation,owner,expected_impact,metric_to_track
0,High,Use forecast-risk categories to guide inventor...,Demand Planning / Inventory Planning,Reduce stockout and excess inventory risk,"WMAPE, stockout rate, excess stock, forecast bias"
1,High,Apply additional stock protection to high-vola...,Demand Planning / Inventory Planning,Improve availability during sudden demand surges,"Peak-to-average ratio, stockout rate, fill rate"
2,Medium,Prepare inventory earlier for the autumn and p...,Demand Planning / Inventory Planning / Purchasing,Improve product availability during peak tradi...,"Peak-season WMAPE, fill rate, stockout rate, i..."


## Business Summary

The forecasting analysis shows that demand planning should not rely on a single model or a single inventory approach across all products.

Forecast accuracy differs by SKU, and products with higher forecast error require more cautious planning. Some products also experience large short-term demand spikes that are difficult for the models to anticipate, creating additional stockout risk.

Seasonality adds another planning consideration, with demand strengthening during the autumn and reaching its highest levels around the pre-Christmas period.

The main planning priorities are therefore to:

1. adjust inventory decisions according to forecast reliability
2. provide additional protection for volatile products
3. prepare inventory earlier for predictable seasonal demand growth

These recommendations can be monitored through forecast accuracy, service level, stockout rate and inventory coverage.